# Модуль 19 — журнал проверки: Hermes и два профиля-роли

Этот ноутбук ничего не устанавливает и не обращается к модели. Он — журнал вашей лабораторки: Hermes вы уже поставили и профили завели по шагам ДЗ (лекция, раздел «Домашнее задание»), а ноутбук проверяет результат subprocess-вызовами `hermes` и чтением файлов `~/.hermes` — и печатает вердикты OK/FAIL.

Что проверяем:

- установка: `hermes version` отвечает;
- профили: worker и watcher существуют, алиас-обёртка на месте;
- изоляция: скиллы и память профилей не пересекаются;
- кроны: у worker `every` + `Repeat: ∞` (не `once in`), у watcher — no-agent вотчдог с реальным файлом скрипта;
- петля обучения: у worker появился хотя бы один local-скилл, записанный самим агентом;
- тикер: heartbeat свежий, отчёты складываются в `cron/output/`.

**Только локальный запуск.** Colab и Kaggle не увидят ваш `~/.hermes` — работа целиком на вашей машине. Все проверки keyless: ни одного LLM-вызова, ни одного ключа.


In [ ]:
# Ячейка 1. Окружение и hermes version
import json, os, shutil, subprocess, time
from datetime import datetime
from pathlib import Path

HERMES = shutil.which("hermes")
HERMES_DIR = Path.home() / ".hermes"
PROFILES = HERMES_DIR / "profiles"

# Если вы назвали профили иначе - поменяйте здесь:
WORKER = "worker"
WATCHER = "watcher"

RESULTS = {}  # копилка вердиктов для итоговой сводки

def verdict(name, ok, detail=""):
    RESULTS[name] = bool(ok)
    print(("OK   - " if ok else "FAIL - ") + name + (f" ({detail})" if detail else ""))
    return bool(ok)

def hermes(*args, timeout=120):
    """Вызов hermes через subprocess; возвращает (код возврата, вывод)."""
    if not HERMES:
        return None, ""
    r = subprocess.run([HERMES, *args], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or "") + (r.stderr or "")

if not HERMES:
    print("FAIL - команда hermes не найдена в PATH. Что делать:")
    print("  1. Hermes ещё не установлен? Официальный инсталлер:")
    print("     curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash")
    print("  2. Установлен, но не виден: бинарь лежит в ~/.local/bin - добавьте каталог в PATH")
    print('     (export PATH="$HOME/.local/bin:$PATH") и перезапустите терминал.')
    print("  3. Jupyter наследует PATH терминала, из которого запущен: перезапустите jupyter")
    print("     из нового терминала и сделайте Restart Kernel.")
    print("  4. Это Colab или Kaggle? Они не подойдут: ноутбук читает ~/.hermes на ВАШЕЙ машине.")
    raise SystemExit("hermes не в PATH - разберитесь с установкой и перезапустите ноутбук")

rc, out = hermes("version")
print(out.strip())
print()
ok = verdict("hermes version отвечает", rc == 0 and "hermes" in out.lower(), HERMES)
if not ok:
    print("Подсказка: начните диагностику с hermes doctor (есть и режим --fix).")
    raise SystemExit("hermes version завершился с ошибкой")


In [ ]:
# Ячейка 2. Профили: worker и watcher существуют
rc, out = hermes("profile", "list")
print(out.strip())
print()

names_ok = verdict("worker и watcher видны в profile list", WORKER in out and WATCHER in out,
                   f"ищем имена: {WORKER}, {WATCHER}")

prof_dirs = sorted(p.name for p in PROFILES.iterdir() if p.is_dir()) if PROFILES.exists() else []
extra = [p for p in prof_dirs if p != "default"]
verdict("минимум два профиля кроме default", len(extra) >= 2, ", ".join(extra) or "профилей нет")

alias = Path.home() / ".local" / "bin" / WORKER
verdict(f"алиас-обёртка ~/.local/bin/{WORKER} существует", alias.exists(),
        "создаётся автоматически при profile create")

if not names_ok:
    print()
    print('Подсказка: hermes profile create worker --description "рабочая роль: ..." ;')
    print("если профили названы иначе - поправьте WORKER/WATCHER в Ячейке 1 и перезапустите ноутбук.")


In [ ]:
# Ячейка 3. Изоляция скиллов: worker против watcher
rc_w, out_w = hermes("-p", WORKER, "skills", "list")
rc_s, out_s = hermes("-p", WATCHER, "skills", "list")

print(f"--- {WORKER} (первые 800 символов) ---")
print(out_w.strip()[:800])
print()
print(f"--- {WATCHER} (первые 800 символов) ---")
print(out_s.strip()[:800])
print()

both = verdict("skills list отвечает у обоих профилей", rc_w == 0 and rc_s == 0)
verdict("наборы скиллов различаются", both and out_w.strip() != out_s.strip(),
        "watcher создан с --no-skills, его список должен быть заметно короче")

marker = PROFILES / WATCHER / ".no-bundled-skills"
if marker.exists():
    print("Бонус: у watcher есть маркер .no-bundled-skills - флаг --no-skills сработал.")


In [ ]:
# Ячейка 4. Изоляция памяти - без LLM, только файлы
# Впишите слово-маркер из факта, который вы записали в память worker (шаг 3 ДЗ),
# например MARKER = "module-19". Пустая строка - проверим только, что память различается.
MARKER = ""

mem_w = PROFILES / WORKER / "memories"
mem_s = PROFILES / WATCHER / "memories"
verdict("каталог memories/ существует у обоих профилей", mem_w.is_dir() and mem_s.is_dir(),
        "память появляется после первого разговора или записи факта")

def read_memories(d):
    if not d.is_dir():
        return ""
    return "\n".join(p.read_text(encoding="utf-8", errors="ignore")
                      for p in sorted(d.glob("*.md")) if p.is_file())

text_w = read_memories(mem_w)
text_s = read_memories(mem_s)

if MARKER:
    verdict("маркер найден в памяти worker", MARKER in text_w, repr(MARKER))
    verdict("маркера нет в памяти watcher", MARKER not in text_s, "память изолирована")
else:
    verdict("память worker и watcher различается", text_w != text_s,
            "для точной проверки впишите MARKER выше")

if not text_w:
    print()
    print("Подсказка: память worker пуста - шаг 3 ДЗ просит записать факт в память worker")
    print("(в чате с worker попросите его запомнить факт) и убедиться, что watcher его не знает.")


In [ ]:
# Ячейка 5. Крон worker: every + Repeat: бесконечность, никаких once in
rc, out = hermes("-p", WORKER, "cron", "list")
print(out.strip())
print()

jobs_path = PROFILES / WORKER / "cron" / "jobs.json"
ok_file = verdict("jobs.json существует", jobs_path.is_file(), str(jobs_path))

jobs = []
if ok_file:
    data = json.loads(jobs_path.read_text(encoding="utf-8"))
    jobs = data.get("jobs", data if isinstance(data, list) else [])

verdict("в jobs.json есть хотя бы один джоб", len(jobs) >= 1,
        "пустой jobs.json при живом тикере - тот самый замаскированный отказ из лекции")

sched_ok = "every" in out and "once in" not in out
verdict("расписание - every, не once in", sched_ok,
        "строка Schedule: once in - неисправность, а не настройка")

rep_ok = bool(jobs) and all((j.get("repeat") or {}).get("times") is None for j in jobs)
verdict("Repeat без лимита (repeat.times == null)", rep_ok,
        "без флага --repeat N у interval-джоба times = null, в cron list это Repeat: беск.")

if not sched_ok:
    print()
    print('Подсказка: hermes -p worker cron create "every 10m" "<промпт>" --name loop ;')
    print("once in-джоб исчезает после срабатывания - цепочка умирает на первом холостом ходе.")


In [ ]:
# Ячейка 6. Вотчдог watcher: no-agent + реальный файл в scripts/
rc, out = hermes("-p", WATCHER, "cron", "list")
print(out.strip())
print()

jobs_path = PROFILES / WATCHER / "cron" / "jobs.json"
jobs = []
if jobs_path.is_file():
    data = json.loads(jobs_path.read_text(encoding="utf-8"))
    jobs = data.get("jobs", data if isinstance(data, list) else [])

verdict("у watcher есть хотя бы один джоб", len(jobs) >= 1)

raw = json.dumps(jobs, ensure_ascii=False)
no_agent_ok = '"no_agent": true' in raw or "no-agent" in out.lower() or "no agent" in out.lower()
verdict("джоб no-agent: LLM не вызывается вовсе", no_agent_ok,
        "скрипт и есть джоб - classic watchdog pattern")

scripts_dir = PROFILES / WATCHER / "scripts"
files = sorted(scripts_dir.iterdir()) if scripts_dir.is_dir() else []
verdict("в scripts/ профиля есть скрипт", len(files) >= 1, str(scripts_dir))

links = [p.name for p in files if p.is_symlink()]
verdict("ни один скрипт не симлинк", bool(files) and not links,
        "симлинки: " + ", ".join(links) if links else "test -L прошёл бы")

if links:
    print()
    print("Подсказка: симлинк тикер отклонит с 'Blocked: script path resolves outside the")
    print("scripts directory', хотя ручной прогон проходит. Скопируйте файл по-настоящему.")


In [ ]:
# Ячейка 7. Петля обучения: worker записал себе скилл
import re

rc, out = hermes("-p", WORKER, "skills", "list")
print("\n".join(out.strip().splitlines()[-3:]))
print()

m = re.search(r"(\d+)\s+local", out)
n_local = int(m.group(1)) if m else 0
verdict("у worker есть хотя бы один local-скилл (шаг «Петля обучения» ДЗ)", n_local > 0,
        f"local-скиллов: {n_local}")
if n_local == 0:
    print("Подсказка: дайте worker одну реальную задачу (worker chat или hermes -p worker -z \"...\")")
    print("и попросите записать решение скиллом - шаг 6 ДЗ в лекции.")

rc2, out2 = hermes("-p", WORKER, "curator", "status")
print()
print("--- curator status worker (первые 600 символов) ---")
print(out2.strip()[:600])


In [ ]:
# Ячейка 8. Здоровье тикера + итоговая сводка
rc, out = hermes("gateway", "list")
print(out.strip())
print()
rc2, out2 = hermes("-p", WORKER, "cron", "status")
print(out2.strip())
print()

now = time.time()
hb = PROFILES / WORKER / "cron" / "ticker_heartbeat"
if hb.is_file():
    age = int(now - hb.stat().st_mtime)
    verdict("тикер worker жив (heartbeat моложе 5 минут)", age < 300, f"{age} с назад")
else:
    verdict("тикер worker жив", False, "нет файла ticker_heartbeat")
    print("Подсказка: тикер живёт в gateway-процессе профиля. hermes -p worker gateway install,")
    print("затем hermes -p worker gateway start (или hermes -p worker gateway run в отдельном терминале),")
    print("затем hermes -p worker cron status; при странностях - hermes doctor (есть и --fix).")

out_dir = PROFILES / WORKER / "cron" / "output"
reports = sorted(out_dir.rglob("*.md"), key=lambda p: p.stat().st_mtime) if out_dir.is_dir() else []
if reports:
    last_age = int(now - reports[-1].stat().st_mtime)
    print(f"Последний отчёт: {reports[-1].name}, {last_age} с назад.")
    print("Лесенка из лекции: живой heartbeat при старых отчётах = 'задания нет', а не 'всё работает'.")
else:
    print("Отчётов в cron/output/ пока нет - если джоб создан только что, дождитесь первого тика")
    print("(тикер проверяет расписание раз в 60 секунд).")

print()
print("=== Итоговая сводка - приложите этот вывод в чат курса как [Модуль 19, ДЗ] ===")
print("Дата прогона:", datetime.now().isoformat(timespec="seconds"))
fails = [k for k, ok in RESULTS.items() if not ok]
for k, ok in RESULTS.items():
    print(("OK   " if ok else "FAIL ") + "- " + k)
total = len(RESULTS)
print()
print(f"Итого: {total - len(fails)}/{total} проверок зелёные"
      + ("" if not fails else " - разберитесь с FAIL выше и перезапустите ноутбук"))


## Итог

Чек-лист перед сдачей — он же критерии приёма из лекции:

- [ ] `hermes version` отрабатывает, в выводе виден метод установки.
- [ ] `hermes profile list` показывает worker и watcher (модель, gateway, алиас), а `hermes profile describe worker` и `hermes profile describe watcher` — разные описания ролей.
- [ ] Скиллы и память профилей изолированы — Ячейки 3–4 зелёные.
- [ ] Крон worker — `every` с `Repeat: ∞`; «once in» в `jobs.json` нет.
- [ ] Вотчдог watcher — no-agent, скрипт лежит реальным файлом в `scripts/` профиля, не симлинком.
- [ ] У worker есть хотя бы один local-скилл, записанный самим агентом, — Ячейка 7 зелёная.
- [ ] `Run all` прошёл целиком, итоговая сводка без FAIL.

Если что-то FAIL — диагностическая лесенка из лекции: `hermes cron list` у профиля → свежесть `cron/ticker_heartbeat` против последнего файла в `cron/output/` → последний `## Response` в отчёте → `hermes gateway list` (тикер живёт в gateway) → `hermes doctor`. Команды и дефолты — по [документации Hermes Agent](https://hermes-agent.nousresearch.com/docs/).

**Что приложить в чат курса:** вывод итоговой сводки из Ячейки 8, пометка `[Модуль 19, ДЗ]`. Преподаватель не нужен: все пункты закрыты — домашка сдана.
